# YOLO Hyperparameter Tuning with Optuna

This notebook provides comprehensive hyperparameter optimization for YOLO models using Optuna.

Features:
- Support for YOLOv8, YOLOv9, YOLOv10, YOLO11, YOLO12
-  Optuna-based hyperparameter optimization
-  Final model training with optimized parameters

Workflow:
1. Install required libraries
2. Configure dataset and model
3. Define hyperparameter search space
4. Run Optuna optimization
5. Visualize results
6. Save best hyperparameters
7. Train final model with optimized settings

## 1. Install Required Libraries

Install all necessary packages for YOLO training and hyperparameter optimization.

In [ ]:
# Install required libraries (uncomment if running in Colab)
# !pip install -q ultralytics optuna plotly kaleido wandb pyyaml

import os
import sys
import gc
import yaml
import json
import torch
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

import wandb

# YOLO and Optuna imports
from ultralytics import YOLO
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

warnings.filterwarnings('ignore')

# Configure matplotlib for notebook display
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 10)

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ Libraries imported successfully')
print(f'✓ Device: {device}')
if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  CUDA Version: {torch.version.cuda}')
    print(f'  Available Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## 2. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Base directories
# Detect environment: Colab or local

IS_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

USE_WANDB = True  # Set to False to disable W&B logging

if IS_COLAB:
    # Running in Google Colab
    BASE_DIR = Path('/computer_vision_yolo')
    
    # Configure W&B API key
    if USE_WANDB:
        
        # In Colab, get API key from secrets
        try:
            from google.colab import userdata
            wandb_api_key = userdata.get('wandb_api_key')
            os.environ['WANDB_API_KEY'] = wandb_api_key
            print('✓ W&B API key loaded from Colab secrets')
        except Exception as e:
            print(f'⚠️  Could not load W&B API key from Colab secrets: {e}')
            print('  To use W&B, add "wandb_api_key" to Colab secrets')
            print('  Secrets icon is in the left sidebar (🔑)')


else:
    # Running locally
    BASE_DIR = Path.cwd().parent
    if USE_WANDB:
        print('✓ Running locally - W&B will use existing login or prompt')

# Model Selection - Choose one of the following:
MODEL_NAME = "yolov10n"

#yolov10n is for testing purpose only
#Mahdy will work yolov8m


# Selected models, to choose from, based on the performance and size:
# YOLOv8:  'yolov8s', 'yolov8m'

# YOLOv10: 'yolov10s', 'yolov10m'

# YOLO12: 'yolo12s'

# Directory structure
MODELS_DIR = BASE_DIR / 'models' / MODEL_NAME
TMP_DIR = BASE_DIR / 'tmp' / MODEL_NAME

# Dataset Selection
# Option 1: Full dataset (~100k images) - for final optimization: "bdd100k_yolo"
# Option 2: Limited dataset (representative samples) - for quick tuning: "bdd100k_yolo_limited"
Dataset_Name = 'bdd100k_yolo_limited'

YOLO_DATASET_ROOT = BASE_DIR / Dataset_Name

# data.yaml path
DATA_YAML_PATH = YOLO_DATASET_ROOT / 'data.yaml'

# Verify dataset exists
if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_YAML_PATH}\n"
        f"Please prepare the dataset first using process_bdd100k_to_yolo_dataset.py"
    )

# Update data.yaml path field for Colab compatibility
import yaml
import tempfile
with open(DATA_YAML_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the 'path' field to use BASE_DIR
data_config['path'] = str(YOLO_DATASET_ROOT)

# Create a temporary data.yaml with corrected paths
temp_data_yaml = TMP_DIR / 'data.yaml'
TMP_DIR.mkdir(parents=True, exist_ok=True)
with open(temp_data_yaml, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)

# Use the temporary data.yaml for training
DATA_YAML_PATH = temp_data_yaml

# Optimization Configuration
N_TRIALS = 5  # Number of optimization trials = 50–70 trials
TIMEOUT_HOURS = 6  # Maximum time for optimization (None for no limit)
N_STARTUP_TRIALS = 1  # Random exploration trials before optimization =10
EPOCHS_PER_TRIAL = 2  # Training epochs per trial = 50
EPOCHS_FINAL_TRAINING = 3  # Training epochs for final model = 150
BATCH_SIZE = 16  # Batch size for training, 32 for T4 GPU, 96 for A100 GPU
IMAGE_SIZE = 640  # Input image size

# Weights & Biases (optional)
USE_WANDB = True  # Set to True to enable W&B logging
WANDB_PROJECT_TUNING = f"yolo-{YOLO_DATASET_ROOT.name}-tuning"
WANDB_PROJECT_TRAINING = f"yolo-{YOLO_DATASET_ROOT.name}-training"

# Generate run identifier
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME_TUNING = f'{MODEL_NAME}_tune_{RUN_TIMESTAMP}'
RUN_NAME_TRAINING = f'{MODEL_NAME}_train_{RUN_TIMESTAMP}'

# Create directories for tuning and training within tune_train folder
# All paths are absolute to ensure consistency across environments (local/Colab)
TUNE_TRAIN_BASE = BASE_DIR / 'tune_train'
TUNE_DIR = TUNE_TRAIN_BASE / 'tune' / RUN_NAME_TUNING
TRAIN_DIR = TUNE_TRAIN_BASE / 'train' / RUN_NAME_TRAINING
TUNE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Keep RUN_DIR for backward compatibility (points to tuning)
RUN_DIR = TUNE_DIR

# Read dataset configuration
NUM_CLASSES = data_config['nc']
CLASS_NAMES = {i: name for i, name in enumerate(data_config['names'])}
CLASS_NAME_TO_ID = {name: i for i, name in enumerate(data_config['names'])}

print('=' * 80)
print('CONFIGURATION SUMMARY')
print('=' * 80)
print(f'Environment: {"Google Colab" if "COLAB_GPU" in os.environ or os.path.exists("/content") else "Local"}')
print(f'Base Directory: {BASE_DIR}')
print(f'Model: {MODEL_NAME}')
print(f'Dataset: {YOLO_DATASET_ROOT.name}')
print(f'Data YAML: {DATA_YAML_PATH}')
print(f'  Dataset path in YAML: {data_config["path"]}')
print(f'Classes: {NUM_CLASSES}')
print(f'Class Names: {CLASS_NAMES}')
print(f'Device: {device}')
print(f'Optimization Trials: {N_TRIALS}')
print(f'Epochs per Trial: {EPOCHS_PER_TRIAL}')
print(f'Epochs Final Training: {EPOCHS_FINAL_TRAINING}')
print(f'Batch Size: {BATCH_SIZE}')
print(f'Image Size: {IMAGE_SIZE}')
print(f'Timeout: {TIMEOUT_HOURS} hours' if TIMEOUT_HOURS else 'No timeout')
print(f'Tuning Directory: {TUNE_DIR}')
print(f'Training Directory: {TRAIN_DIR}')
if USE_WANDB:
    print(f'W&B Logging: Enabled')
    print(f'  Tuning Project: {WANDB_PROJECT_TUNING}')
    print(f'  Training Project: {WANDB_PROJECT_TRAINING}')
else:
    print(f'W&B Logging: Disabled')
print('=' * 80)

## 3. Load Base YOLO Model

In [ ]:
# Load YOLO model with automatic download
model_path = MODELS_DIR / f'{MODEL_NAME}.pt'

if not model_path.exists():
    print(f'Model not found at {model_path}')
    print(f'Downloading {MODEL_NAME} ...')
    
    try:
        # Download model - it will be cached by ultralytics
        MODEL_NAME_n = MODEL_NAME 
        if MODEL_NAME.startswith('yolov11') or MODEL_NAME.startswith('yolov12'):
            MODEL_NAME_n = MODEL_NAME + '.pt'
        model = YOLO(MODEL_NAME_n)
        
        # Create models directory
        MODELS_DIR.mkdir(parents=True, exist_ok=True)
        
        # Save model to our directory using export/save
        try:
            # Try to save using the model's save method
            if hasattr(model, 'save'):
                model.save(str(model_path))
                print(f'✓ Model downloaded and saved to {model_path}')
                print(f'  Size: {model_path.stat().st_size / (1024*1024):.1f} MB')
            else:
                # Fallback: copy from cache
                cache_patterns = [
                    str(Path.home() / '.cache' / 'ultralytics' / '**' / f'{MODEL_NAME}.pt'),
                    str(Path.home() / '.config' / 'Ultralytics' / '**' / f'{MODEL_NAME}.pt'),
                ]
                
                model_found = False
                for pattern in cache_patterns:
                    cache_paths = glob.glob(pattern, recursive=True)
                    if cache_paths:
                        shutil.copy(cache_paths[0], model_path)
                        print(f'✓ Model downloaded and saved to {model_path}')
                        print(f'  Size: {model_path.stat().st_size / (1024*1024):.1f} MB')
                        model_found = True
                        break
                
                if not model_found:
                    print(f'✓ Model loaded from ultralytics cache')
                    print(f'  Note: Model is in cache, not copied to {model_path}')
                    print(f'  This is normal and the model will work correctly')
        except Exception as save_error:
            print(f'⚠️  Could not save model to custom location: {save_error}')
            print(f'✓ Model loaded successfully from ultralytics cache')
            
    except Exception as e:
        print(f'\n❌ Error downloading model: {e}')
        raise
else:
    print("test")
    model = YOLO(str(model_path))
    print(f'✓ Model loaded from {model_path}')

# Get model information
model_info={}
info=model.info()
keys=["layers","params","size(MB)","FLOPs(G)"]

for x,y in zip(keys,info):
    model_info[x] = y
    
model_params=model_info.get("params",0)
model_size_mb=model_info.get("size(MB)",0)
flops_gflops=model_info.get("FLOPs(G)",0)


print(f'\n📊 Model Information:')
print(f'  Model: {MODEL_NAME}')
print(f'  Classes in model: {len(model.names)}')
print(f'  Task: {model.task}')
print(f'  Parameters: {model_params / 1e6:.1f}M')
print(f'  Model Size: {model_size_mb:.1f} MB')
print(f'  FLOPs (640x640): {flops_gflops:.2f} GFLOPs')

## 4. Verify Dataset Structure

In [ ]:
# ============================================================================
# VERIFY DATASET STRUCTURE
# ============================================================================

print('Verifying YOLO dataset structure...')
print(f'\n📁 Dataset Root: {YOLO_DATASET_ROOT}')

# Check all splits
dataset_stats = {}
for split in ['train', 'val', 'test']:
    images_dir = YOLO_DATASET_ROOT / 'images' / split
    labels_dir = YOLO_DATASET_ROOT / 'labels' / split
    
    if images_dir.exists() and labels_dir.exists():
        num_images = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
        num_labels = len(list(labels_dir.glob('*.txt')))
        dataset_stats[split] = {'images': num_images, 'labels': num_labels}
        print(f'  ✓ {split:5s}: {num_images:6d} images, {num_labels:6d} labels')
    else:
        print(f'  ⚠️  {split:5s}: Directory not found')
        dataset_stats[split] = {'images': 0, 'labels': 0}

print(f'\n📄 Configuration: {DATA_YAML_PATH}')
print(f'  Classes: {NUM_CLASSES}')
print(f'  Names: {CLASS_NAMES}')

total_images = sum(stats['images'] for stats in dataset_stats.values())
print(f'\n✓ Dataset verified: {total_images:,} total images')
print('✓ Ready for hyperparameter optimization')

## 5. Define Hyperparameter Search Space

In [ ]:
# ============================================================================
# DEFINE OPTIMIZED HYPERPARAMETER SEARCH SPACE (BEST FOR LARGE DATASETS)
# ============================================================================

def define_hyperparameters(trial):
    """
    Best-practice hyperparameter search for YOLO (optimized for large datasets).
    
    - Focus only on high-impact parameters
    - Avoid low-impact or unstable augmentations
    - Tightened ranges for faster convergence
    """

    # ---------------------------
    # 1) Optimizer + Learning Rate
    # ---------------------------
    optimizer_choice = trial.suggest_categorical('optimizer', ['SGD', 'AdamW'])

    lr0 = trial.suggest_float('lr0', 1e-4, 5e-3, log=True)  # stable range
    lrf = trial.suggest_float('lrf', 0.1, 0.5)  # LR decay factor

    # ---------------------------
    # 2) Regularization
    # ---------------------------
    momentum = trial.suggest_float('momentum', 0.85, 0.97)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    
    # -------------------------------
    # WARMUP HYPERPARAMETERS
    # -------------------------------
    warmup_epochs = trial.suggest_int("warmup_epochs", 0, 3)       # Number of warmup epochs (0–3)
    warmup_momentum = trial.suggest_float("warmup_momentum", 0.5, 0.95)  # Start momentum
    warmup_bias_lr = trial.suggest_float("warmup_bias_lr", 0.0, 0.1)     # Start bias learning rate


    # ---------------------------
    # 3) Light & Stable Augmentation (best for big datasets)
    # ---------------------------
    hsv_s = trial.suggest_float('hsv_s', 0.4, 0.8)
    hsv_v = trial.suggest_float('hsv_v', 0.4, 0.8)

    scale = trial.suggest_float('scale', 0.7, 1.3)
    translate = trial.suggest_float('translate', 0.0, 0.2)

    mosaic = trial.suggest_float('mosaic', 0.5, 1.0)  # strong mosaic improves generalization

    # ---------------------------
    # 4) Loss balancing
    # ---------------------------
    box = trial.suggest_float('box', 4.0, 10.0)
    cls = trial.suggest_float('cls', 0.5, 2.0)
    dfl = trial.suggest_float('dfl', 0.5, 2.0)

    # ---------------------------
    # 5) Compile parameters
    # ---------------------------
    hyperparams = {
        # Optimizer
        'optimizer': optimizer_choice,

        # LR
        'lr0': lr0,
        'lrf': lrf,

        # Regularization
        'momentum': momentum,
        'weight_decay': weight_decay,

        # Mild augmentation
        'hsv_s': hsv_s,
        'hsv_v': hsv_v,
        'scale': scale,
        'translate': translate,
        'mosaic': mosaic,

        # Loss weights
        'box': box,
        'cls': cls,
        'dfl': dfl,

        # Fixed parameters
        'epochs': EPOCHS_PER_TRIAL,
        'batch': BATCH_SIZE,
        'imgsz': IMAGE_SIZE,
        'device': device,
        'val': True, # Enable validation during training
        'patience': 20,  # Early stopping patience
        'save': True,  # Save intermediate models
        'plots': True,  # Generate plots for each trial
        'cache': True,  # Cache images for faster training
        'workers': 8,  # Number of data loading workers
        'close_mosaic': 10,  # Disable mosaic in last N epochs
        'verbose': True,  # Reduce verbosity
    }

    return hyperparams


print('✓ Hyperparameter search space defined')
print('\n📊 Search Space Summary:')
print('  Strategy: Using wide ranges, letting Optuna find optimal values')
print('  Optimizers: SGD, Adam, AdamW')
print('  Learning Rates: Wide range for exploration')
print('  Augmentation: Full range (0-1 for probabilities)')
print('  Loss Weights: Wide range for different dataset characteristics')
print(f'  Fixed: epochs={EPOCHS_PER_TRIAL}, batch={BATCH_SIZE}, imgsz={IMAGE_SIZE}')

## 6. Define Objective Function

In [ ]:
# DEFINE OBJECTIVE FUNCTION FOR OPTUNA
# ============================================================================

def objective(trial):
    """Objective function for Optuna hyperparameter optimization.

    Steps:
    1. Sample hyperparameters for the current trial
    2. Train a YOLO model with those hyperparameters
    3. Evaluate the model on the validation set
    4. Return validation mAP@0.5 (to maximize)
    """
    # Get hyperparameters for this trial
    hyperparameters = define_hyperparameters(trial)

    # Create trial-specific directory (absolute path under BASE_DIR)
    trial_dir = TUNE_DIR / f"trial_{trial.number:03d}"
    trial_dir.mkdir(exist_ok=True, parents=True)

    # Initialize W&B if enabled
    wandb_run = None
    if USE_WANDB:
        try:
            os.environ['WANDB_DIR'] = str(trial_dir)
            wandb_run = wandb.init(
                project=WANDB_PROJECT_TUNING,
                name=f'{MODEL_NAME}_trial_{trial.number:03d}',
                config=hyperparameters,
                dir=str(trial_dir),
                reinit=True
            )
        except Exception as e:
            print(f'⚠️  W&B initialization failed: {e}')
            wandb_run = None

    # Print trial information
    print(f"\n{'=' * 80}")
    print(f"TRIAL {trial.number}/{N_TRIALS}")
    print(f"{'=' * 80}")
    print(f"Optimizer: {hyperparameters['optimizer']}")
    print(f"Learning Rate: lr0={hyperparameters['lr0']:.6f}, lrf={hyperparameters['lrf']:.4f}")
    print(f"Momentum: {hyperparameters['momentum']:.4f}, Weight Decay: {hyperparameters['weight_decay']:.6f}")
    print(f"Warmup: epochs={hyperparameters['warmup_epochs']}, momentum={hyperparameters['warmup_momentum']:.2f}, bias_lr={hyperparameters['warmup_bias_lr']:.2f}")
    print(
        "Augmentation: "
        f"hsv_h={hyperparameters['hsv_h']:.3f}, hsv_s={hyperparameters.get('hsv_s',0):.3f}, hsv_v={hyperparameters.get('hsv_v',0):.3f}, "
        f"translate={hyperparameters['translate']:.3f}, scale={hyperparameters['scale']:.3f}, "
        f"mosaic={hyperparameters['mosaic']:.2f}, mixup={hyperparameters['mixup']:.2f}"
    )
    print(
        "Loss Weights: "
        f"box={hyperparameters['box']:.2f}, cls={hyperparameters['cls']:.2f}, dfl={hyperparameters['dfl']:.2f}"
    )
    print(f"{'=' * 80}")

    trial_model = None
    map50 = 0.001  # Default penalty for failed trials
    
    try:
        # Load fresh model for this trial
        trial_model = YOLO(str(model_path))
        
        # Train model with hyperparameters and W&B support
        trial_run_name = f"{MODEL_NAME}_trial_{trial.number:03d}"
        train_results = trial_model.train(
            data=str(DATA_YAML_PATH),
            project=str(trial_dir),
            name=trial_run_name,
            exist_ok=True,
            wandb=USE_WANDB,
            **hyperparameters,
        )
        
        # Validate model
        validation_results = trial_model.val(
            data=str(DATA_YAML_PATH),
            split="val",
            project=str(trial_dir),
            name="val",
            verbose=False,
            wandb=USE_WANDB,
        )

        # Extract metrics
        map50 = float(validation_results.box.map50)
        map50_95 = float(validation_results.box.map)
        precision = float(validation_results.box.mp)
        recall = float(validation_results.box.mr)
        
        # Save training metrics if available
        train_metrics = {}
        if hasattr(train_results, 'results_dict'):
            train_metrics = {key: float(value) if isinstance(value, (int,float,np.floating,np.integer)) else value
                             for key,value in train_results.results_dict.items()
                             if key not in ['fitness']}

        # Save trial results JSON
        trial_results = {
            "trial_number": trial.number,
            "model_name": MODEL_NAME,
            "dataset": YOLO_DATASET_ROOT.name,
            "trial_directory": str(trial_dir),
            "hyperparameters": {k: float(v) if isinstance(v,(np.floating,np.integer)) else v for k,v in hyperparameters.items()},
            "validation_metrics": {"map50": map50, "map50_95": map50_95, "precision": precision, "recall": recall},
            "training_metrics": train_metrics,
            "training_config": {
                "epochs": EPOCHS_PER_TRIAL,
                "batch_size": BATCH_SIZE,
                "image_size": IMAGE_SIZE,
                "device": device,
            },
            "timestamp": datetime.now().isoformat(),
            "status": "completed"
        }
        results_path = trial_dir / "trial_results.json"
        with open(results_path, "w") as f:
            json.dump(trial_results, f, indent=2)
        print(f"✓ Trial {trial.number} completed, results saved: {results_path}")
        print(f"  mAP@0.5: {map50:.4f}")
        print(f"  mAP@0.5:0.95: {map50_95:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")

        # Remove last.pt to save space
        last_pt = trial_dir / trial_run_name / "weights/last.pt"
        if last_pt.exists():
            last_pt.unlink()
            print("🧹 Removed last.pt to save space")

    except Exception as e:
        print(f"❌ Trial {trial.number} failed: {error}")
        import traceback
        traceback.print_exc()
        error_results = {
            "trial_number": trial.number,
            "model_name": MODEL_NAME,
            "dataset": YOLO_DATASET_ROOT.name,
            "trial_directory": str(trial_dir),
            "hyperparameters": {
                key: float(value)
                if isinstance(value, (np.floating, np.integer)) else value
                for key, value in hyperparameters.items()
            },
            "error": str(error),
            "error_type": type(error).__name__,
            "timestamp": datetime.now().isoformat(),
            "status": "failed"
        }
        with open(trial_dir / "trial_results.json", "w") as f:
            json.dump(error_results, f, indent=2)

    finally:
        if trial_model:
            del trial_model
            print("🧹 Model deleted from memory")
        if wandb_run:
            wandb.finish()
        gc.collect()
        if device=="cuda":
            torch.cuda.empty_cache()
            print("🧹 CUDA cache cleared")

    return map50

print("✓ Objective function defined")

## 7. Run Hyperparameter Optimization

In [ ]:
# RUN HYPERPARAMETER OPTIMIZATION WITH OPTUNA
# ============================================================================

print('\n' + '=' * 80)
print('STARTING HYPERPARAMETER OPTIMIZATION')
print('=' * 80)
print(f'Model: {MODEL_NAME}')
print(f'Dataset: {YOLO_DATASET_ROOT.name}')
print(f'Number of Trials: {N_TRIALS}')
print(f'Epochs per Trial: {EPOCHS_PER_TRIAL}')
print(f'Timeout: {TIMEOUT_HOURS} hours' if TIMEOUT_HOURS else 'No timeout')
print(f'Device: {device}')
print('=' * 80)

# Create Optuna study
study = optuna.create_study(
    study_name=f'{MODEL_NAME}_optuna_{RUN_TIMESTAMP}',
    direction='maximize',  # Maximize mAP@0.5
    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=N_STARTUP_TRIALS,  # Random trials before optimization
        multivariate=True,  # Consider parameter interactions
        group=True  # Group related parameters
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=N_STARTUP_TRIALS,
        n_warmup_steps=15,  # Wait before pruning
        interval_steps=5  # Check every 5 steps
    )
)

# Run optimization
start_time = datetime.now()
print(f'\n🚀 Optimization started at {start_time.strftime("%Y-%m-%d %H:%M:%S")}')

try:
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        timeout=TIMEOUT_HOURS * 3600 if TIMEOUT_HOURS else None,
        show_progress_bar=True,
        callbacks=[
            lambda study, trial: print(f'\n✓ Completed {len(study.trials)}/{N_TRIALS} trials'),
            lambda study, trial: gc.collect()  # Force garbage collection after each trial
        ]
    )
except KeyboardInterrupt:
    print('\n⚠️  Optimization interrupted by user')
except Exception as e:
    print(f'\n❌ Optimization failed: {e}')
    import traceback
    traceback.print_exc()

end_time = datetime.now()
duration = end_time - start_time

print('\n' + '=' * 80)
print('OPTIMIZATION COMPLETED')
print('=' * 80)
print(f'Started: {start_time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Ended: {end_time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Duration: {duration}')
print(f'Total Trials: {len(study.trials)}')
print(f'Completed Trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'Pruned Trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}')
print(f'Failed Trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])}')
print(f'\nBest Trial: {study.best_trial.number}')
print(f'Best mAP@0.5: {study.best_value:.4f}')
print('=' * 80)

## 7.5. Save All Trials Summary

In [ ]:
# SAVE CONSOLIDATED SUMMARY OF ALL TRIALS
# ============================================================================

print('\n' + '=' * 80)
print('SAVING CONSOLIDATED TRIAL SUMMARY')
print('=' * 80)

# Collect all trial results dynamically from study
all_trials_data = []

for trial in study.trials:
    trial_dir = TUNE_DIR / f"trial_{trial.number:03d}"
    results_file = trial_dir / "trial_results.json"
    
    if results_file.exists():
        try:
            with open(results_file, 'r') as f:
                trial_data = json.load(f)
                all_trials_data.append(trial_data)
        except Exception as e:
            print(f"⚠️  Could not read trial {trial.number} results: {e}")
    else:
        print(f"⚠️  No results file found for trial {trial.number}")

# Create comprehensive summary
optimization_summary = {
    "model_name": MODEL_NAME,
    "dataset": YOLO_DATASET_ROOT.name,
    "optimization_config": {
        "n_trials": N_TRIALS,
        "epochs_per_trial": EPOCHS_PER_TRIAL,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "timeout_hours": TIMEOUT_HOURS,
        "n_startup_trials": N_STARTUP_TRIALS,
    },
    "optimization_results": {
        "start_time": start_time.isoformat(),
        "end_time": end_time.isoformat(),
        "duration_seconds": duration.total_seconds(),
        "total_trials": len(study.trials),
        "completed_trials": len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]),
        "pruned_trials": len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
        "failed_trials": len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL]),
        "best_trial_number": study.best_trial.number,
        "best_map50": study.best_value,
    },
    "best_hyperparameters": study.best_params,
    "all_trials": all_trials_data,
    "timestamp": datetime.now().isoformat(),
}

# Save consolidated summary as JSON
summary_path = TUNE_DIR / f"{MODEL_NAME}_all_trials_summary.json"
with open(summary_path, 'w') as f:
    json.dump(optimization_summary, f, indent=2)

print(f'✓ Consolidated JSON summary saved: {summary_path}')
print(f'  Total trials saved: {len(all_trials_data)}')

# Create CSV summary for easy analysis
csv_data = []
for trial_data in all_trials_data:
    row = {
        'trial_number': trial_data.get('trial_number'),
        'status': trial_data.get('status'),
        'map50': trial_data.get('validation_metrics', {}).get('map50'),
        'map50_95': trial_data.get('validation_metrics', {}).get('map50_95'),
        'precision': trial_data.get('validation_metrics', {}).get('precision'),
        'recall': trial_data.get('validation_metrics', {}).get('recall'),
        'error_type': trial_data.get('error_type', '')  # Include error type if failed
    }
    # Add hyperparameters
    for key, value in trial_data.get('hyperparameters', {}).items():
        row[f'hp_{key}'] = value
    # Flag best trial
    row['best_trial'] = trial_data.get('trial_number') == study.best_trial.number
    csv_data.append(row)

df_trials = pd.DataFrame(csv_data)

# Sort CSV by mAP@0.5 descending (best first)
df_trials.sort_values(by='map50', ascending=False, inplace=True)

# Save CSV
csv_path = TUNE_DIR / f"{MODEL_NAME}_all_trials_summary.csv"
df_trials.to_csv(csv_path, index=False)

print(f'✓ CSV summary saved: {csv_path}')
print(f'  Columns: {len(df_trials.columns)}, Rows: {len(df_trials)}')
print('=' * 80)

# Display summary statistics
if len(df_trials) > 0:
    print('\n📊 Trial Summary Statistics:')
    print(f'  Completed Trials: {len(df_trials[df_trials["status"] == "completed"])}')
    print(f'  Failed Trials: {len(df_trials[df_trials["status"] == "failed"])}')
    
    completed_trials = df_trials[df_trials['status'] == 'completed']
    if len(completed_trials) > 0:
        best_trial_row = completed_trials.loc[completed_trials["map50"].idxmax()]
        print(f'\n  mAP@0.5 Statistics:')
        print(f'    Best: {best_trial_row["map50"]:.4f} (Trial {best_trial_row["trial_number"]})')
        print(f'    Worst: {completed_trials["map50"].min():.4f}')
        print(f'    Mean: {completed_trials["map50"].mean():.4f}')
        print(f'    Std: {completed_trials["map50"].std():.4f}')
        print(f'    Median: {completed_trials["map50"].median():.4f}')
print('=' * 80)

## 8. Analyze Best Hyperparameters

In [ ]:
# EXTRACT AND DISPLAY BEST HYPERPARAMETERS
# ============================================================================

print('\n' + '=' * 80)
print('BEST HYPERPARAMETERS')
print('=' * 80)

best_params = study.best_params
best_trial = study.best_trial

print(f'\nBest Trial Number: {best_trial.number}')
print(f'Best mAP@0.5: {study.best_value:.4f}')
print('\nOptimized Hyperparameters:')
print(json.dumps(best_params, indent=2))

# Save best parameters to JSON
best_params_json = TUNE_DIR / 'best_hyperparameters.json'
with open(best_params_json, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'dataset': str(YOLO_DATASET_ROOT),
        'best_trial': best_trial.number,
        'best_map50': study.best_value,
        'total_trials': len(study.trials),
        'hyperparameters': best_params,
        'optimization_config': {
            'n_trials': N_TRIALS,
            'epochs_per_trial': EPOCHS_PER_TRIAL,
            'batch_size': BATCH_SIZE,
            'image_size': IMAGE_SIZE,
        },
        'timestamp': datetime.now().isoformat()
    }, f, indent=2)

print(f'\n✓ Best hyperparameters saved to: {best_params_json}')

# Save to YAML format (ready for YOLO training)
best_params_yaml = TUNE_DIR / 'best_hparams.yaml'
with open(best_params_yaml, 'w') as f:
    yaml.dump(best_params, f, default_flow_style=False, sort_keys=False)

print(f'✓ Best hyperparameters saved to: {best_params_yaml}')
print('=' * 80)

## 9. Visualize Optimization Results

In [ ]:
# ============================================================================
# VISUALIZE OPTIMIZATION RESULTS: HISTORY, PARAMETER IMPORTANCE, SLICE PLOTS
# ============================================================================

from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice
from datetime import datetime

print('\n' + '=' * 80)
print('GENERATING OPTIMIZATION VISUALIZATIONS')
print('=' * 80)

if len(study.trials) == 0:
    print("⚠️  No trials found in study, skipping visualization.")
else:
    timestamp_str = datetime.now().strftime('%Y%m%d_%H%M%S')

    # -----------------------------
    # 1️⃣ Optimization History Plot
    # -----------------------------
    try:
        print('\n📈 Creating optimization history plot...')
        fig_history = plot_optimization_history(study)
        fig_history.update_layout(
            title=f'{MODEL_NAME} - Hyperparameter Optimization History',
            xaxis_title='Trial Number',
            yaxis_title='mAP@0.5',
            template='plotly_white',
            width=1200,
            height=600
        )
        fig_history.show()

        # Save HTML
        optimization_history_path = TUNE_DIR / f'optimization_history_{timestamp_str}.html'
        fig_history.write_html(str(optimization_history_path))
        print(f'✓ HTML saved to: {optimization_history_path}')

        # Save PNG (if Kaleido available)
        optimization_history_img = TUNE_DIR / f'optimization_history_{timestamp_str}.png'
        try:
            fig_history.write_image(str(optimization_history_img), width=1200, height=600, scale=2)
            print(f'✓ PNG saved to: {optimization_history_img}')
        except Exception as e:
            print(f'ℹ️  Could not save PNG (kaleido not available): {e}')

    except Exception as e:
        print(f'❌ Failed to create optimization history plot: {e}')

    # -----------------------------
    # 2️⃣ Parameter Importance Plot
    # -----------------------------
    try:
        print('\n📊 Creating parameter importance plot...')
        fig_importance = plot_param_importances(study)
        fig_importance.update_layout(
            title=f'{MODEL_NAME} - Hyperparameter Importance',
            xaxis_title='Importance',
            yaxis_title='Parameter',
            template='plotly_white',
            width=1200,
            height=800
        )
        fig_importance.show()

        # Save HTML
        param_importance_path = TUNE_DIR / f'parameter_importance_{timestamp_str}.html'
        fig_importance.write_html(str(param_importance_path))
        print(f'✓ HTML saved to: {param_importance_path}')

        # Save PNG
        try:
            param_importance_img = TUNE_DIR / f'parameter_importance_{timestamp_str}.png'
            fig_importance.write_image(str(param_importance_img), width=1200, height=800, scale=2)
            print(f'✓ PNG saved to: {param_importance_img}')
        except Exception as e:
            print(f'ℹ️  Could not save PNG: {e}')
            param_importance_img = None

    except (RuntimeError, ValueError) as e:
        print(f'⚠️  Could not generate parameter importance plot: {e}')
        print('  (This can happen when trials have insufficient data variation)')
        param_importance_img = None

    # -----------------------------
    # 3️⃣ Parameter Slice Plots
    # -----------------------------
    try:
        print('\n🔍 Creating parameter slice plots...')
        fig_slice = plot_slice(study)
        fig_slice.update_layout(
            title=f'{MODEL_NAME} - Parameter Slice Plot',
            template='plotly_white',
            width=1400,
            height=1000
        )
        fig_slice.show()

        # Save HTML
        slice_path = TUNE_DIR / f'parameter_slice_{timestamp_str}.html'
        fig_slice.write_html(str(slice_path))
        print(f'✓ HTML saved to: {slice_path}')

        # Save PNG
        try:
            slice_img_path = TUNE_DIR / f'parameter_slice_{timestamp_str}.png'
            fig_slice.write_image(str(slice_img_path), width=1400, height=1000, scale=2)
            print(f'✓ PNG saved to: {slice_img_path}')
        except Exception as e:
            print(f'ℹ️  Could not save PNG: {e}')

    except Exception as e:
        print(f'⚠️  Could not generate parameter slice plot: {e}')

print('=' * 80)
print('✅ Optimization visualizations completed')


## 10. Create Results Summary

In [ ]:
# CREATE TRIALS SUMMARY
# ============================================================================

print('\n' + '=' * 80)
print('TRIALS SUMMARY')
print('=' * 80)

# Compile all trial data
trials_data = []
for trial in study.trials:
    trial_info = {
        'trial': trial.number,
        'mAP@0.5': trial.value if trial.value else 0.0,
        'state': trial.state.name,
        'duration_seconds': (trial.datetime_complete - trial.datetime_start).total_seconds() if trial.datetime_complete else None,
    }
    # Add all parameters
    trial_info.update(trial.params)
    trials_data.append(trial_info)

# Create DataFrame
df_trials = pd.DataFrame(trials_data)

# Sort by performance
df_trials_sorted = df_trials.sort_values('mAP@0.5', ascending=False)

print('\n📊 TOP 10 TRIALS:')
print('=' * 80)
# Display top 10 with selected columns
display_cols = ['trial', 'mAP@0.5', 'state', 'optimizer', 'lr0', 'momentum', 'weight_decay', 'mixup']
available_cols = [col for col in display_cols if col in df_trials_sorted.columns]
print(df_trials_sorted[available_cols].head(10).to_string(index=False))
print('=' * 80)

# Save complete trials summary
trials_csv_path = TUNE_DIR / 'trials_summary.csv'
df_trials_sorted.to_csv(trials_csv_path, index=False)
print(f'\n✓ Complete trials summary saved to: {trials_csv_path}')

# Save study object
study_path = TUNE_DIR / 'optuna_study.pkl'
import pickle
with open(study_path, 'wb') as f:
    pickle.dump(study, f)
print(f'✓ Optuna study object saved to: {study_path}')

print('=' * 80)

## 11. Save Hyperparameters for Training

In [ ]:
# PREPARE FINAL TRAINING CONFIGURATION
# ============================================================================

print('\n' + '=' * 80)
print('PREPARING FINAL TRAINING CONFIGURATION')
print('=' * 80)

# Prepare final training hyperparameters
final_training_params = best_params.copy()
final_training_params.update({
    # Extended training settings
    'epochs': EPOCHS_FINAL_TRAINING,  # Full training epochs
    'batch': BATCH_SIZE,
    'imgsz': IMAGE_SIZE,
    'device': device,
    
    # Training control
    'patience': 25,  # Early stopping patience
    'save': True,  # Save models
    'save_period': 10,  # Save checkpoint every N epochs
    'plots': True,  # Generate training plots
    'verbose': True,  # Detailed output
    
    # Efficiency
    'cache': True,  # Cache images
    'workers': 8,  # Data loading workers
    'amp': True,  # Automatic mixed precision
    
    # Validation
    'val': True,  # Run validation
    
    # Project organization - use absolute paths for Colab compatibility
    'project': str(TRAIN_DIR),
    'name': f'{MODEL_NAME}_finetuned',
    'exist_ok': True,
})

# Save training configuration
training_config_path = TRAIN_DIR / f'{MODEL_NAME}_finetuned_config.yaml'
with open(training_config_path, 'w') as f:
    yaml.dump(final_training_params, f, default_flow_style=False, sort_keys=False)

print(f'\n✓ Training configuration saved to: {training_config_path}')

# Also save as JSON with metadata
training_config_json = TRAIN_DIR / f'{MODEL_NAME}_finetuned_config.json'
with open(training_config_json, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'base_model_path': str(model_path),
        'dataset_root': str(YOLO_DATASET_ROOT),
        'data_yaml_path': str(DATA_YAML_PATH),
        'optimization_results': {
            'best_trial': study.best_trial.number,
            'best_map50': study.best_value,
            'total_trials': len(study.trials),
            'optimization_duration': str(duration),
        },
        'hyperparameters': final_training_params,
        'timestamp': datetime.now().isoformat(),
        'notes': 'Use these hyperparameters for full model training with 100 epochs'
    }, f, indent=2)

print(f'✓ Training configuration (with metadata) saved to: {training_config_json}')

print('\n📋 Training Configuration Summary:')
print(f'  Epochs: {final_training_params["epochs"]}')
print(f'  Batch Size: {final_training_params["batch"]}')
print(f'  Image Size: {final_training_params["imgsz"]}')
print(f'  Optimizer: {final_training_params["optimizer"]}')
print(f'  Learning Rate: {final_training_params["lr0"]:.6f}')
print(f'  Device: {final_training_params["device"]}')

print('=' * 80)

## 12. Train Final Model with Optimized Hyperparameters

Now train the final model using the best hyperparameters found during optimization.

In [ ]:
# TRAIN FINAL MODEL WITH OPTIMIZED HYPERPARAMETERS (ENHANCED)
# ============================================================================
print('\n' + '=' * 80)
print('TRAINING FINAL MODEL WITH OPTIMIZED HYPERPARAMETERS')
print('=' * 80)

# Load fresh model
print(f'\n📦 Loading base model: {MODEL_NAME}')
final_model = YOLO(str(model_path))

print(f'\n🚀 Starting final training with best hyperparameters...')
print(f'  Epochs: {final_training_params["epochs"]}')
print(f'  Dataset: {DATA_YAML_PATH}')
print(f'  Device: {device}')
print('\nThis may take a while. Training progress will be displayed below.')
print('=' * 80)


# Generate model timestamp for naming
model_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f"{MODEL_NAME}_finetuned_{model_timestamp}"

# Add reproducibility: optional random seed
final_training_params.setdefault('seed', 42)

# Train model with optimized hyperparameters
final_results = final_model.train(
    data=str(DATA_YAML_PATH),
    project=str(TRAIN_DIR),
    name=run_name,
    val=True,
    verbose=True,
    save=True,
    wandb=USE_WANDB,
    **final_training_params,
)

# Finish W&B run safely
if USE_WANDB:
    try:
        wandb.finish()
    except Exception as e:
        print(f'⚠️ Could not finish W&B run: {e}')
        
print('\n' + '=' * 80)
print('FINAL TRAINING COMPLETED')
print('=' * 80)

# Get final validation metrics
final_val_results = final_model.val(
    data=str(DATA_YAML_PATH),
    project=str(TRAIN_DIR),
    name='val',
    wandb=USE_WANDB,
)

final_metrics = {
    'map50': float(final_val_results.box.map50),
    'map50_95': float(final_val_results.box.map),
    'precision': float(final_val_results.box.mp),
    'recall': float(final_val_results.box.mr),
}

print('\n📊 Final Model Performance:')
print(f"  mAP@0.5: {final_metrics['map50']:.4f}")
print(f"  mAP@0.5:0.95: {final_metrics['map50_95']:.4f}")
print(f"  Precision: {final_metrics['precision']:.4f}")
print(f"  Recall: {final_metrics['recall']:.4f}")



# Compare with best trial from tuning
improvement = final_metrics['map50'] - study.best_value
print('\n📈 Improvement vs Best Trial:')
print(f"  Best Trial mAP@0.5: {study.best_value:.4f}")
print(f"  Final Model mAP@0.5: {final_metrics['map50']:.4f}")
print(f"  Improvement: {improvement:+.4f} ({improvement/study.best_value*100:+.2f}%)")

# Save finetuned model
finetuned_model_name = f"{MODEL_NAME}_finetuned-{model_timestamp}"
finetuned_model_path = MODELS_DIR / MODEL_NAME / f"{finetuned_model_name}.pt"
final_model.save(str(finetuned_model_path))
print(f"💾 Finetuned model saved: {finetuned_model_path}")

# Save metadata with full config
model_metadata = {
    'model_name': MODEL_NAME,
    'finetuned_name': finetuned_model_name,
    'timestamp': model_timestamp,
    'dataset': YOLO_DATASET_ROOT.name,
    'best_hyperparameters': study.best_params,
    'final_training_params': final_training_params,
    'best_trial_map50': float(study.best_value),
    'final_metrics': final_metrics,
    'improvement': float(improvement),
    'epochs_trained': final_training_params['epochs'],
}

metadata_path = MODELS_DIR / MODEL_NAME / f"{finetuned_model_name}_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)
print(f"💾 Model metadata saved: {metadata_path}")


## 13. Save Final Model

In [ ]:
# SAVE FINAL OPTIMIZED MODEL
# ============================================================================

print('\n' + '=' * 80)
print('SAVING FINAL OPTIMIZED MODEL')
print('=' * 80)

# Generate model filename with timestamp
model_timestamp = datetime.now().strftime('%Y%m%d')
final_model_name = f'{MODEL_NAME}_finetuned_{model_timestamp}.pt'
final_model_path = MODELS_DIR / final_model_name

# Copy best weights to models directory
weights_path = TRAIN_DIR / 'runs' / f'{MODEL_NAME}_finetuned' / 'weights' / 'best.pt'
if weights_path.exists():
    shutil.copy(weights_path, final_model_path)
    print(f'\n✓ Final model saved to: {final_model_path}')
    print(f'  Size: {final_model_path.stat().st_size / (1024*1024):.1f} MB')
else:
    print(f'\n⚠️  Best weights not found at: {weights_path}')
    print('  Model may still be in training directory')

# Save model metadata
metadata = {
    'model_name': MODEL_NAME,
    'model_path': str(final_model_path),
    'dataset': str(YOLO_DATASET_ROOT),
    'training_date': datetime.now().isoformat(),
    'optimization': {
        'n_trials': len(study.trials),
        'best_trial': study.best_trial.number,
        'trial_map50': study.best_value,
        'optimization_duration': str(duration),
    },
    'hyperparameters': best_params,
    'final_metrics': final_metrics,
    'training_config': {
        'epochs': final_training_params['epochs'],
        'batch_size': final_training_params['batch'],
        'image_size': final_training_params['imgsz'],
    }
}

metadata_path = MODELS_DIR / f'{MODEL_NAME}_finetuned_{model_timestamp}_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✓ Model metadata saved to: {metadata_path}')
print('=' * 80)

## 14. Test Final Model

In [ ]:
# RUN FINAL VALIDATION ON TEST SET (ENHANCED)
# ============================================================================
print('\n' + '=' * 80)
print('RUNNING FINAL VALIDATION ON TEST SET')
print('=' * 80)

results_summary = []
IOU_THRESHOLDS = 0.5  # Could expand to [0.5, 0.55, 0.6] if needed
finetuned_model_name = MODEL_NAME

# Add YOLO test scripts path safely
scrpt_dir = BASE_DIR / "yolo_test"
if str(scrpt_dir) not in sys.path:
    sys.path.append(str(scrpt_dir))

try:
    from run_yolo_validation_report import run_validation_pipeline

    result = run_validation_pipeline(
        model_name=finetuned_model_name,
        dataset_name=Dataset_Name,
        split="test",
        iou_threshold=IOU_THRESHOLDS,
        base_dir=BASE_DIR,
        use_wandb=True,
        save_reports=True,
        batch_size=BATCH_SIZE,
    )
    
    overall = result["metrics"]["overall"]
    yolo_overall = result["metrics"]["yolo_metrics"]
    
    results_summary.append({
        "model_name": finetuned_model_name,
        "dataset": Dataset_Name,
        "split": "test",
        "iou": IOU_THRESHOLDS,
        "precision_confusion": overall["precision"],
        "recall_confusion": overall["recall"],
        "f1_confusion": overall["f1"],
        "precision_yolo": yolo_overall["precision"],
        "recall_yolo": yolo_overall["recall"],
        "map50": yolo_overall["map50"],
        "map50_95": yolo_overall["map50_95"],
        "params_m": result["model_info"]["params"] / 1e6,
        "size_mb": result["model_info"]["size(MB)"],
        "fps": result["metrics"]["fps"],
        "status": "ok",
        "run_dir": str(result["run_dir"]),
        "hyperparameters": final_training_params,  # traceable
    })
    
except Exception as e:
    print(f"⚠️ Model {finetuned_model_name} failed during validation: {e}")
    results_summary.append({
        "model_name": finetuned_model_name,
        "dataset": Dataset_Name,
        "split": "test",
        "iou": IOU_THRESHOLDS,
        "status": "error",
        "error_message": str(e)
    })

# Convert to DataFrame
results_df = pd.DataFrame(results_summary)
print('\n📊 Final Validation Results:')
display(results_df)


## 15. Generate Training PDF Report

In [ ]:
# GENERATE TRAINING PDF REPORT
# ============================================================================

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors as rl_colors
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from PIL import Image as PILImage

print('\n' + '=' * 80)
print('GENERATING TRAINING PDF REPORT')
print('=' * 80)

# Create training PDF report
pdf_training_report_path = TRAIN_DIR / f'{MODEL_NAME}_training_report.pdf'

doc = SimpleDocTemplate(str(pdf_training_report_path), pagesize=A4,
                       rightMargin=30, leftMargin=30,
                       topMargin=30, bottomMargin=30)

story = []
styles = getSampleStyleSheet()

# Custom styles
title_style = ParagraphStyle(
    'CustomTitle',
    parent=styles['Heading1'],
    fontSize=24,
    textColor=rl_colors.HexColor('#2c3e50'),
    spaceAfter=30,
    alignment=TA_CENTER
)

heading_style = ParagraphStyle(
    'CustomHeading',
    parent=styles['Heading2'],
    fontSize=16,
    textColor=rl_colors.HexColor('#34495e'),
    spaceAfter=12,
    spaceBefore=20
)

# Title
story.append(Paragraph(f'{MODEL_NAME} Final Training Report', title_style))
story.append(Spacer(1, 12))

# Configuration info
info_data = [
    ['Model:', MODEL_NAME],
    ['Dataset:', YOLO_DATASET_ROOT.name],
    ['Training Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
    ['Training Epochs:', str(EPOCHS_FINAL_TRAINING)],
    ['Batch Size:', str(BATCH_SIZE)],
    ['Image Size:', str(IMAGE_SIZE)]
]

info_table = Table(info_data, colWidths=[2.2*inch, 3.8*inch])
info_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), rl_colors.HexColor('#ecf0f1')),
    ('TEXTCOLOR', (0, 0), (-1, -1), rl_colors.black),
    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
    ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
    ('TOPPADDING', (0, 0), (-1, -1), 8),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.white)
]))
story.append(info_table)
story.append(Spacer(1, 20))

# Dataset Information
story.append(Paragraph('Dataset Information', heading_style))

dataset_info_data = [
    ['Property', 'Value'],
    ['Dataset', YOLO_DATASET_ROOT.name],
    ['Number of Classes', str(NUM_CLASSES)],
    ['Classes', ', '.join(CLASS_NAMES)],
    ['Data YAML', str(DATA_YAML_PATH.name)],
]

dataset_info_table = Table(dataset_info_data, colWidths=[2*inch, 4*inch])
dataset_info_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#16a085')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (0, -1), 'LEFT'),
    ('ALIGN', (1, 0), (1, -1), 'LEFT'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 11),
    ('FONTSIZE', (0, 1), (-1, -1), 9),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black),
    ('VALIGN', (0, 0), (-1, -1), 'TOP'),
]))
story.append(dataset_info_table)
story.append(Spacer(1, 20))

# Optimization Summary
story.append(Paragraph('Optimization Summary', heading_style))

opt_summary_data = [
    ['Metric', 'Value'],
    ['Total Trials', str(len(study.trials))],
    ['Best Trial Number', str(study.best_trial.number)],
    ['Best Trial mAP@0.5', f"{study.best_value:.4f}"],
    ['Epochs per Trial', str(EPOCHS_PER_TRIAL)],
    ['Final Training Epochs', str(EPOCHS_FINAL_TRAINING)],
]

opt_summary_table = Table(opt_summary_data, colWidths=[3*inch, 3*inch])
opt_summary_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#f39c12')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('FONTSIZE', (0, 1), (-1, -1), 10),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(opt_summary_table)
story.append(Spacer(1, 20))

# Optimized hyperparameters used
story.append(PageBreak())
story.append(Paragraph('Optimized Hyperparameters Used', heading_style))

hyperparam_data = [['Parameter', 'Value']]
for key, value in best_params.items():
    hyperparam_data.append([key, f'{value:.6f}' if isinstance(value, float) else str(value)])

hyperparam_table = Table(hyperparam_data, colWidths=[3*inch, 3*inch])
hyperparam_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#3498db')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('FONTSIZE', (0, 1), (-1, -1), 9),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(hyperparam_table)
story.append(Spacer(1, 20))

# Final model performance
if 'final_metrics' in globals():
    story.append(PageBreak())
    story.append(Paragraph('Final Model Performance', heading_style))
    
    final_perf_data = [
        ['Metric', 'Value'],
        ['mAP@0.5', f"{final_metrics['map50']:.4f}"],
        ['mAP@0.5:0.95', f"{final_metrics['map50_95']:.4f}"],
        ['Precision', f"{final_metrics['precision']:.4f}"],
        ['Recall', f"{final_metrics['recall']:.4f}"],
    ]
    
    final_perf_table = Table(final_perf_data, colWidths=[3*inch, 3*inch])
    final_perf_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#27ae60')),
        ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 12),
        ('FONTSIZE', (0, 1), (-1, -1), 11),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
        ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
    ]))
    story.append(final_perf_table)
    story.append(Spacer(1, 20))
    
    # Comparison with trial performance
    story.append(Paragraph('Performance Comparison', heading_style))
    improvement = final_metrics['map50'] - study.best_value
    
    comparison_data = [
        ['Stage', 'mAP@0.5'],
        ['Best Tuning Trial', f"{study.best_value:.4f}"],
        ['Final Training', f"{final_metrics['map50']:.4f}"],
        ['Improvement', f"{improvement:+.4f}"]
    ]
    
    comparison_table = Table(comparison_data, colWidths=[3*inch, 3*inch])
    comparison_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#e74c3c')),
        ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 12),
        ('FONTSIZE', (0, 1), (-1, -1), 11),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
        ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
    ]))
    story.append(comparison_table)
    story.append(Spacer(1, 20))

# Training configuration details
story.append(PageBreak())
story.append(Paragraph('Training Configuration', heading_style))

training_config_data = [
    ['Parameter', 'Value'],
    ['Epochs', str(final_training_params['epochs'])],
    ['Batch Size', str(final_training_params['batch'])],
    ['Image Size', str(final_training_params['imgsz'])],
    ['Optimizer', final_training_params['optimizer']],
    ['Learning Rate (lr0)', f"{final_training_params['lr0']:.6f}"],
    ['Learning Rate Final (lrf)', f"{final_training_params['lrf']:.4f}"],
    ['Momentum', f"{final_training_params['momentum']:.4f}"],
    ['Weight Decay', f"{final_training_params['weight_decay']:.6f}"],
    ['Early Stopping Patience', str(final_training_params['patience'])],
    ['AMP Enabled', str(final_training_params['amp'])],
    ['Cache Enabled', str(final_training_params['cache'])],
]

training_config_table = Table(training_config_data, colWidths=[3*inch, 3*inch])
training_config_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#9b59b6')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('FONTSIZE', (0, 1), (-1, -1), 9),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(training_config_table)
story.append(Spacer(1, 20))

# Footer
story.append(Spacer(1, 30))
story.append(Paragraph('Generated by YOLO Training Notebook',
                      ParagraphStyle('Footer', parent=styles['Normal'],
                                   alignment=TA_CENTER, textColor=rl_colors.grey)))
story.append(Paragraph('BDD100K Dataset - Computer Vision Project',
                      ParagraphStyle('Footer2', parent=styles['Normal'],
                                   alignment=TA_CENTER, textColor=rl_colors.grey)))

# Build PDF
try:
    doc.build(story)
    print(f'\n✓ Training PDF report generated: {pdf_training_report_path}')
    print(f'  Size: {pdf_training_report_path.stat().st_size / 1024:.2f} KB')
except Exception as e:
    print(f'\n❌ Error generating training PDF: {e}')
    import traceback
    traceback.print_exc()

print('=' * 80)

## 16. Generate PDF Report

Create a comprehensive PDF report with optimization results, visualizations, and model performance.

In [ ]:
# GENERATE PDF REPORT
# ============================================================================

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors as rl_colors
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from PIL import Image as PILImage

print('\n' + '=' * 80)
print('GENERATING TUNING PDF REPORT')
print('=' * 80)

# Create tuning PDF report
pdf_report_path = TUNE_DIR / f'{MODEL_NAME}_tuning_report.pdf'

doc = SimpleDocTemplate(str(pdf_report_path), pagesize=A4,
                       rightMargin=30, leftMargin=30,
                       topMargin=30, bottomMargin=30)

story = []
styles = getSampleStyleSheet()

# Custom styles
title_style = ParagraphStyle(
    'CustomTitle',
    parent=styles['Heading1'],
    fontSize=24,
    textColor=rl_colors.HexColor('#2c3e50'),
    spaceAfter=30,
    alignment=TA_CENTER
)

heading_style = ParagraphStyle(
    'CustomHeading',
    parent=styles['Heading2'],
    fontSize=16,
    textColor=rl_colors.HexColor('#34495e'),
    spaceAfter=12,
    spaceBefore=20
)

# Title
story.append(Paragraph(f'{MODEL_NAME} Hyperparameter Tuning Report', title_style))
story.append(Spacer(1, 12))

# Configuration info
info_data = [
    ['Model:', MODEL_NAME],
    ['Dataset:', YOLO_DATASET_ROOT.name],
    ['Timestamp:', datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
    ['Total Trials:', str(len(study.trials))],
    ['Best Trial:', str(study.best_trial.number)],
    ['Best mAP@0.5:', f'{study.best_value:.4f}']
]

info_table = Table(info_data, colWidths=[2.2*inch, 3.8*inch])
info_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), rl_colors.HexColor('#ecf0f1')),
    ('TEXTCOLOR', (0, 0), (-1, -1), rl_colors.black),
    ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
    ('FONTNAME', (0, 0), (0, -1), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 10),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
    ('TOPPADDING', (0, 0), (-1, -1), 8),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.white)
]))
story.append(info_table)
story.append(Spacer(1, 20))

# Optimization Configuration
story.append(Paragraph('Optimization Configuration', heading_style))

opt_config_data = [
    ['Parameter', 'Value'],
    ['Total Trials', str(N_TRIALS)],
    ['Epochs per Trial', str(EPOCHS_PER_TRIAL)],
    ['Batch Size', str(BATCH_SIZE)],
    ['Image Size', str(IMAGE_SIZE)],
    ['Startup Trials', str(N_STARTUP_TRIALS)],
    ['Device', device],
    ['Dataset Path', str(YOLO_DATASET_ROOT)],
    ['Number of Classes', str(NUM_CLASSES)],
]

opt_config_table = Table(opt_config_data, colWidths=[3*inch, 3*inch])
opt_config_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#95a5a6')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('FONTSIZE', (0, 1), (-1, -1), 9),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(opt_config_table)
story.append(Spacer(1, 20))

# Trial Statistics
story.append(Paragraph('Trial Statistics', heading_style))

completed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
pruned_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
failed_trials = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

trial_stats_data = [
    ['Status', 'Count'],
    ['Completed', str(completed_trials)],
    ['Pruned', str(pruned_trials)],
    ['Failed', str(failed_trials)],
    ['Total', str(len(study.trials))],
]

trial_stats_table = Table(trial_stats_data, colWidths=[3*inch, 3*inch])
trial_stats_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#16a085')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('FONTSIZE', (0, 1), (-1, -1), 10),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(trial_stats_table)
story.append(Spacer(1, 20))

# Best hyperparameters
story.append(PageBreak())
story.append(Paragraph('Best Hyperparameters', heading_style))

hyperparam_data = [['Parameter', 'Value']]
for key, value in best_params.items():
    hyperparam_data.append([key, f'{value:.6f}' if isinstance(value, float) else str(value)])

hyperparam_table = Table(hyperparam_data, colWidths=[3*inch, 3*inch])
hyperparam_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#3498db')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 12),
    ('FONTSIZE', (0, 1), (-1, -1), 9),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(hyperparam_table)
story.append(Spacer(1, 20))

# Top 10 trials
story.append(PageBreak())
story.append(Paragraph('Top 10 Trials', heading_style))

top10_data = [['Trial', 'mAP@0.5', 'State']]
for _, row in df_trials_sorted.head(10).iterrows():
    top10_data.append([
        str(int(row['trial'])),
        f"{row['mAP@0.5']:.4f}",
        row['state']
    ])

top10_table = Table(top10_data, colWidths=[1.5*inch, 2*inch, 2.5*inch])
top10_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#27ae60')),
    ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 11),
    ('FONTSIZE', (0, 1), (-1, -1), 9),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ('TOPPADDING', (0, 0), (-1, -1), 6),
    ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
    ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
]))
story.append(top10_table)
story.append(Spacer(1, 20))

# Optimization history
story.append(PageBreak())
story.append(Paragraph('Optimization History', heading_style))
story.append(Spacer(1, 12))

optimization_history_img = TUNE_DIR / 'optimization_history.png'
if optimization_history_img.exists():
    try:
        with PILImage.open(optimization_history_img) as img:
            img_width, img_height = img.size
            aspect_ratio = img_height / img_width
            pdf_width = 6.5 * inch
            pdf_height = pdf_width * aspect_ratio
            if pdf_height > 7 * inch:
                pdf_height = 7 * inch
                pdf_width = pdf_height / aspect_ratio
            story.append(Image(str(optimization_history_img), width=pdf_width, height=pdf_height))
    except Exception as e:
        print(f'⚠️  Could not load optimization history: {e}')
        story.append(Paragraph('Optimization history chart not available.', styles['Normal']))
else:
    story.append(Paragraph('Optimization history chart not found (PNG format required).', styles['Normal']))

story.append(Spacer(1, 20))

# Parameter importance
story.append(PageBreak())
story.append(Paragraph('Parameter Importance', heading_style))
story.append(Spacer(1, 12))

param_importance_img = TUNE_DIR / 'parameter_importance.png'
if param_importance_img.exists():
    try:
        with PILImage.open(param_importance_img) as img:
            img_width, img_height = img.size
            aspect_ratio = img_height / img_width
            pdf_width = 6.5 * inch
            pdf_height = pdf_width * aspect_ratio
            if pdf_height > 7 * inch:
                pdf_height = 7 * inch
                pdf_width = pdf_height / aspect_ratio
            story.append(Image(str(param_importance_img), width=pdf_width, height=pdf_height))
    except Exception as e:
        print(f'⚠️  Could not load parameter importance: {e}')
        story.append(Paragraph('Parameter importance chart not available.', styles['Normal']))
else:
    story.append(Paragraph('Parameter importance chart not available or could not be generated.', styles['Normal']))

story.append(Spacer(1, 20))

# Final model performance (if available)
if 'final_metrics' in globals():
    story.append(PageBreak())
    story.append(Paragraph('Final Model Performance', heading_style))
    
    final_perf_data = [
        ['Metric', 'Value'],
        ['mAP@0.5', f"{final_metrics['map50']:.4f}"],
        ['mAP@0.5:0.95', f"{final_metrics['map50_95']:.4f}"],
        ['Precision', f"{final_metrics['precision']:.4f}"],
        ['Recall', f"{final_metrics['recall']:.4f}"],
    ]
    
    final_perf_table = Table(final_perf_data, colWidths=[3*inch, 3*inch])
    final_perf_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), rl_colors.HexColor('#e74c3c')),
        ('TEXTCOLOR', (0, 0), (-1, 0), rl_colors.whitesmoke),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE', (0, 0), (-1, 0), 12),
        ('FONTSIZE', (0, 1), (-1, -1), 11),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 8),
        ('TOPPADDING', (0, 0), (-1, -1), 8),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [rl_colors.white, rl_colors.lightgrey]),
        ('GRID', (0, 0), (-1, -1), 1, rl_colors.black)
    ]))
    story.append(final_perf_table)
    story.append(Spacer(1, 20))

# Footer
story.append(Spacer(1, 30))
story.append(Paragraph('Generated by YOLO Hyperparameter Tuning Notebook',
                      ParagraphStyle('Footer', parent=styles['Normal'],
                                   alignment=TA_CENTER, textColor=rl_colors.grey)))
story.append(Paragraph('BDD100K Dataset - Computer Vision Project',
                      ParagraphStyle('Footer2', parent=styles['Normal'],
                                   alignment=TA_CENTER, textColor=rl_colors.grey)))

# Build PDF
try:
    doc.build(story)
    print(f'\n✓ Tuning PDF report generated: {pdf_report_path}')
    print(f'  Size: {pdf_report_path.stat().st_size / 1024:.2f} KB')
except Exception as e:
    print(f'\n❌ Error generating tuning PDF: {e}')
    import traceback
    traceback.print_exc()

print('=' * 80)

## 13. Final summary


In [ ]:
# FINAL SUMMARY
# ============================================================================

print('\n\n')
print('=' * 80)
print('HYPERPARAMETER OPTIMIZATION COMPLETE!')
print('=' * 80)

print(f'\n📊 Project: {MODEL_NAME} on {YOLO_DATASET_ROOT.name}')
print(f'📅 Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

print(f'\n🔬 Optimization Summary:')
print(f'  Total Trials: {len(study.trials)}')
print(f'  Completed: {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'  Best Trial: {study.best_trial.number}')
print(f'  Best Trial mAP@0.5: {study.best_value:.4f}')
print(f'  Duration: {duration}')

if 'final_metrics' in globals():
    print(f'\n🎯 Final Model Performance:')
    print(f'  mAP@0.5: {final_metrics["map50"]:.4f}')
    print(f'  mAP@0.5:0.95: {final_metrics["map50_95"]:.4f}')
    print(f'  Precision: {final_metrics["precision"]:.4f}')
    print(f'  Recall: {final_metrics["recall"]:.4f}')

print(f'\n📁 Generated Files:')
print(f'\n  📊 Tuning Results (in {TUNE_DIR}):')
print(f'    - best_hyperparameters.json')
print(f'    - best_hparams.yaml')
print(f'    - trials_summary.csv')
print(f'    - optuna_study.pkl')
print(f'  📈 Tuning Visualizations:')
print(f'    - optimization_history.html / .png')
print(f'    - parameter_importance.html / .png')
print(f'    - parameter_slice.html / .png')
print(f'  📄 Tuning PDF Report:')
print(f'    - {MODEL_NAME}_tuning_report.pdf')
print(f'\n  🎯 Training Results (in {TRAIN_DIR}):')
print(f'    - {MODEL_NAME}_finetuned_config.yaml')
print(f'    - {MODEL_NAME}_finetuned_config.json')
print(f'  📄 Training PDF Report:')
print(f'    - {MODEL_NAME}_training_report.pdf')

if 'final_model_path' in globals():
    print(f'  🎯 Final Model:')
    print(f'    - {final_model_path}')
    print(f'    - {metadata_path}')
    print(f'  ⚙️  Training Config:')
    print(f'    - {training_config_path}')
    print(f'    - {training_config_json}')

print(f'\n📂 All results saved to:')
print(f'  Tuning: {TUNE_DIR}')
print(f'  Training: {TRAIN_DIR}')

print(f'\n🎓 Top 5 Hyperparameters (by importance):')
try:
    importances = optuna.importance.get_param_importances(study)
    for i, (param, importance) in enumerate(list(importances.items())[:5], 1):
        print(f'  {i}. {param}: {importance:.4f}')
except:
    print('  (Not available - requires completed trials with variation)')

print(f'\n🚀 Next Steps:')
print(f'  1. Review tuning PDF report: {TUNE_DIR / f"{MODEL_NAME}_tuning_report.pdf"}')
print(f'  2. Review training PDF report: {TRAIN_DIR / f"{MODEL_NAME}_training_report.pdf"}')
print(f'  3. Review optimization visualizations in: {TUNE_DIR}')
if 'final_model_path' in globals():
    print(f'  4. Use final model for inference: {final_model_path}')
    print(f'  5. Check training plots in: {TRAIN_DIR / "runs" / f"{MODEL_NAME}_finetuned"}')
else:
    print(f'  4. Run final training section to create finetuned model')
print(f'  6. Consider testing different model sizes (yolov8s, yolov8m, etc.)')
print(f'  7. Evaluate on test set for final performance metrics')

print('\n' + '=' * 80)
print('SUCCESS! ✓')
print('=' * 80)